## VectorStores And Retrievers

This tutorial will familarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support the retrieval of data -- from (vector) databases and other sources -- for integrations with LLM workflows. They are important for applications that fetch data to be reasoned over as a part of model inference, as in case of Retrieval-Augmented Generation.

We will cover
- Document
- VectorStores
- Retrievers

### Documents

LangChain implements a Document abstractions, which is intented to represent a unit of text and associated metadata. It has two attributes:-

- page_content:- a string representing the content
- metadata:- a dict containing arbitrary metadata

The metadata attribute can capture information about the sources of the documents, it's relationships to the other documents, and other informations. Note that an individual Document object often represents a chunk of larger documents.

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by large language models.",
        metadata={"source": "langchain_docs"}
    ),
    Document(
        page_content="FAISS is a vector database library developed by Meta for efficient similarity search.",
        metadata={"source": "faiss_docs"}
    ),
    Document(
        page_content="Ollama allows developers to run open-source large language models locally on their machines.",
        metadata={"source": "ollama_docs"}
    ),
    Document(
        page_content="RAG stands for Retrieval Augmented Generation and combines retrieval with language generation.",
        metadata={"source": "rag_docs"}
    ),
    Document(
        page_content="LangSmith helps developers trace, monitor, evaluate, and debug LLM applications.",
        metadata={"source": "langsmith_docs"}
    ),
    Document(
        page_content="Prompt engineering involves designing instructions that guide a language model's behavior.",
        metadata={"source": "prompt_engineering_docs"}
    )
]

In [2]:
documents

[Document(metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications powered by large language models.'),
 Document(metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.'),
 Document(metadata={'source': 'ollama_docs'}, page_content='Ollama allows developers to run open-source large language models locally on their machines.'),
 Document(metadata={'source': 'rag_docs'}, page_content='RAG stands for Retrieval Augmented Generation and combines retrieval with language generation.'),
 Document(metadata={'source': 'langsmith_docs'}, page_content='LangSmith helps developers trace, monitor, evaluate, and debug LLM applications.'),
 Document(metadata={'source': 'prompt_engineering_docs'}, page_content="Prompt engineering involves designing instructions that guide a language model's behavior.")]

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x16cdbd120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x16cdbd780>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
## We are going to use HuggingFace for an open source embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6281.36it/s]


In [5]:
## VectorStores

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings_model)
vectorstore


In [6]:
vectorstore.similarity_search("vector")

[Document(id='35b6468d-9235-4ac1-af13-09c4f6f33815', metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.'),
 Document(id='d55471cb-a5d9-420a-b773-e17098fa2c70', metadata={'source': 'langsmith_docs'}, page_content='LangSmith helps developers trace, monitor, evaluate, and debug LLM applications.'),
 Document(id='f3802082-768b-47bf-8b47-e3ab7e002bd5', metadata={'source': 'rag_docs'}, page_content='RAG stands for Retrieval Augmented Generation and combines retrieval with language generation.'),
 Document(id='da8f8454-1e04-47e2-b162-d4b4b9d0bc98', metadata={'source': 'ollama_docs'}, page_content='Ollama allows developers to run open-source large language models locally on their machines.')]

### What is an Async Query?

Normally when you query a vector database:

results = db.similarity_search("What is LangChain?")

your program waits until the search finishes.

Send Query
    ↓
Wait...
    ↓
Get Results
    ↓
Continue Execution

This is called synchronous execution.

#### Async Version

results = await db.asimilarity_search(
    "What is LangChain?"
)

Now Python can do other work while waiting for the search to complete.

Send Query
    ↓
Search Running
    ↓
Do Other Tasks
    ↓
Get Results Later

In [7]:
## Async query

await vectorstore.asimilarity_search("vector")

[Document(id='35b6468d-9235-4ac1-af13-09c4f6f33815', metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.'),
 Document(id='d55471cb-a5d9-420a-b773-e17098fa2c70', metadata={'source': 'langsmith_docs'}, page_content='LangSmith helps developers trace, monitor, evaluate, and debug LLM applications.'),
 Document(id='f3802082-768b-47bf-8b47-e3ab7e002bd5', metadata={'source': 'rag_docs'}, page_content='RAG stands for Retrieval Augmented Generation and combines retrieval with language generation.'),
 Document(id='da8f8454-1e04-47e2-b162-d4b4b9d0bc98', metadata={'source': 'ollama_docs'}, page_content='Ollama allows developers to run open-source large language models locally on their machines.')]

In [8]:
## Similarity search with score

vectorstore.similarity_search_with_score("vector")

[(Document(id='35b6468d-9235-4ac1-af13-09c4f6f33815', metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.'),
  1.5428577661514282),
 (Document(id='d55471cb-a5d9-420a-b773-e17098fa2c70', metadata={'source': 'langsmith_docs'}, page_content='LangSmith helps developers trace, monitor, evaluate, and debug LLM applications.'),
  1.7813704013824463),
 (Document(id='f3802082-768b-47bf-8b47-e3ab7e002bd5', metadata={'source': 'rag_docs'}, page_content='RAG stands for Retrieval Augmented Generation and combines retrieval with language generation.'),
  1.8508093357086182),
 (Document(id='da8f8454-1e04-47e2-b162-d4b4b9d0bc98', metadata={'source': 'ollama_docs'}, page_content='Ollama allows developers to run open-source large language models locally on their machines.'),
  1.92116379737854)]

### Retriever

Langchain vectorstores donot subclass Runnable, and so cannot be integrated into Langchain Expression Language Chains.

Langchain Retrievers are Runnables, so they implement a certain set of methods (e.g. synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this overselves, without subclassing Retriever. If we choose what method we have to use to retrieve documents, we can create a runnable easily. Below we will build around the similarity_search method:

#### Two methods to create the Retriever

1. To use the RunnableLambda --> this will apply the similarity_search method to all the parameters that we passed in the batch method, here k = 1 means retrieve only the top result.

In [9]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

## here the similarity_search will be applied to both "vector" and "framework" and the top result for both the search will be returned.
retriever = RunnableLambda(vectorstore.similarity_search).bind(k = 1)
retriever.batch(["vector", "framework"])

[[Document(id='35b6468d-9235-4ac1-af13-09c4f6f33815', metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.')],
 [Document(id='0ebbfd44-8cc5-4d01-8832-2c5b683d89e0', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications powered by large language models.')]]

2. as_retriever method --> this method will generate a Retirever, specifically a VectorStoreRetriever. These Retriever includes specific search_type and search_kwargs attributes that identify what methods of the underlying vectorstore to call, and how to parameterize them. For example we can replicate the above with the following:

In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["vector", "framework"])

[[Document(id='35b6468d-9235-4ac1-af13-09c4f6f33815', metadata={'source': 'faiss_docs'}, page_content='FAISS is a vector database library developed by Meta for efficient similarity search.')],
 [Document(id='0ebbfd44-8cc5-4d01-8832-2c5b683d89e0', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications powered by large language models.')]]

In [14]:
## integrate retriever with the chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = '''
    Answer the following question based on the provided context only.
    
    {question}
    
    Context: 
    {context}

'''

prompt = ChatPromptTemplate.from_messages([("human", message)])
rag_cahin = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm
response = rag_cahin.invoke("tell me about Framework")

response.content


'Based on the provided context, LangChain is a framework for building applications powered by large language models.'